# 🩸 Hemalyzer - ConvNeXt Cell Classifier API Server

This notebook runs the ConvNeXt blood cell classification model on Google Colab with GPU acceleration and exposes it via ngrok for the Render backend to access.

## Setup Instructions:
1. Upload your `best_leukemia_model.pth` to Google Drive
2. Get a free ngrok auth token from https://ngrok.com/
3. Run all cells in order
4. Copy the ngrok URL and set it as `COLAB_MODEL_URL` in your Render backend environment

**Note:** Keep this notebook running during your demo. The ngrok URL changes each session.

## 1️⃣ Install Dependencies

In [ ]:
# Install required packages
!pip install flask flask-cors pyngrok torch torchvision opencv-python-headless pillow -q
print("✅ Dependencies installed!")

## 2️⃣ Mount Google Drive & Load Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set the path to your model file in Google Drive
# Update this path to match your model location!
MODEL_PATH = '/content/drive/MyDrive/Hemalyzer/best_leukemia_model.pth'

import os
if os.path.exists(MODEL_PATH):
    print(f"✅ Model found at: {MODEL_PATH}")
else:
    print(f"❌ Model NOT found at: {MODEL_PATH}")
    print("Please update MODEL_PATH to point to your model file in Google Drive")

## 3️⃣ Configure ngrok Authentication

In [ ]:
# Get your free auth token from: https://dashboard.ngrok.com/get-started/your-authtoken
# Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual token

NGROK_AUTH_TOKEN = '39BUmcuHTVyQZ0bOxSMSihGOoZS_5o7a4vrWHLYbNBQtm2kUM'  # <-- REPLACE THIS!

from pyngrok import ngrok

# Set the auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok authenticated!")

## 4️⃣ Define the ConvNeXt Classifier (Preprocessing + Model)

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from torchvision.models import convnext_base
import base64
import io

# ============================================================
# ADAPTIVE CELL PREPROCESSING (Matches Training Pipeline)
# ============================================================
class AdaptiveCellPreprocessing:
    """
    Adaptive preprocessing matching training pipeline:
    1. Stain normalization (OD space normalization)
    2. CLAHE in YUV space (clipLimit=3.0, tileGridSize=8x8)
    3. Cell detection and centering (Otsu thresholding + contour detection)
    """

    def __init__(self, target_size=384, normalize_staining=True, detect_cell=True, fast_mode=False):
        self.target_size = target_size
        self.normalize_staining = normalize_staining
        self.detect_cell = detect_cell
        self.fast_mode = fast_mode
        self.clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        self.morph_kernel = np.ones((3, 3), np.uint8)

    def __call__(self, img):
        """Apply preprocessing to PIL Image"""
        img_array = np.array(img)

        if self.fast_mode:
            img_array = cv2.resize(img_array, (self.target_size, self.target_size))
            img_yuv = cv2.cvtColor(img_array, cv2.COLOR_RGB2YUV)
            img_yuv[:, :, 0] = self.clahe.apply(img_yuv[:, :, 0])
            img_array = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)
            return Image.fromarray(img_array)

        if self.normalize_staining:
            img_array = self._normalize_staining(img_array)

        img_array = self._adaptive_histogram_equalization(img_array)

        if self.detect_cell:
            img_array = self._detect_and_center_cell(img_array)
        else:
            img_array = cv2.resize(img_array, (self.target_size, self.target_size))

        return Image.fromarray(img_array)

    def _normalize_staining(self, img_array):
        """Normalize H&E staining using OD space approach"""
        img_float = img_array.astype(np.float32) / 255.0
        img_float = np.maximum(img_float, 1e-6)
        od = -np.log(img_float)
        od_norm = od / (np.percentile(od, 99, axis=(0, 1), keepdims=True) + 1e-6)
        od_norm = np.clip(od_norm, 0, 1)
        img_normalized = (255 * np.exp(-od_norm)).astype(np.uint8)
        return img_normalized

    def _adaptive_histogram_equalization(self, img_array):
        """Apply CLAHE to luminance channel in YUV space"""
        img_yuv = cv2.cvtColor(img_array, cv2.COLOR_RGB2YUV)
        img_yuv[:, :, 0] = self.clahe.apply(img_yuv[:, :, 0])
        img_enhanced = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)
        return img_enhanced

    def _detect_and_center_cell(self, img_array):
        """Detect cell and center it using Otsu thresholding"""
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, self.morph_kernel, iterations=2)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, self.morph_kernel, iterations=1)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(largest_contour)
            cx, cy = x + w // 2, y + h // 2
            pad = int(max(w, h) * 0.7) + 20
            y1 = max(0, cy - pad)
            y2 = min(img_array.shape[0], cy + pad)
            x1 = max(0, cx - pad)
            x2 = min(img_array.shape[1], cx + pad)
            cell_crop = img_array[y1:y2, x1:x2]
            cell_crop = cv2.resize(cell_crop, (self.target_size, self.target_size))
            return cell_crop

        return cv2.resize(img_array, (self.target_size, self.target_size))


# ============================================================
# CONVNEXT CLASSIFIER
# ============================================================
class ConvNeXtClassifier:
    """ConvNeXt classifier for blood cell classification"""

    def __init__(self):
        self.model = None
        self.class_names = None
        self.sickle_cell_class_idx = None
        self.device = None
        self.transform = None
        self.preprocessor = None
        self.sickle_cell_confidence_threshold = 0.875

    def load_model(self, model_path):
        """Load ConvNeXt model for classification"""
        try:
            # Use GPU if available
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            print(f"Using device: {self.device}")

            # Load checkpoint
            checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)

            # Get class info
            if isinstance(checkpoint, dict) and 'num_classes' in checkpoint:
                num_classes = checkpoint['num_classes']
                self.class_names = checkpoint.get('class_names', [])
            else:
                state_dict = checkpoint if not isinstance(checkpoint, dict) else checkpoint.get('model_state_dict', checkpoint)
                classifier_weight_key = 'classifier.2.weight' if 'classifier.2.weight' in state_dict else 'classifier.3.weight'
                num_classes = state_dict[classifier_weight_key].shape[0]
                self.class_names = []

            print(f"Loading ConvNeXt Base model with {num_classes} classes...")
            print(f"Class names: {self.class_names}")

            # Initialize model
            model = convnext_base(weights=None)

            # Detect classifier structure
            state_dict = checkpoint if not isinstance(checkpoint, dict) else checkpoint.get('model_state_dict', checkpoint)
            classifier_layer_idx = 0
            for key in state_dict.keys():
                if key.startswith('classifier.') and key.endswith('.weight'):
                    parts = key.split('.')
                    if len(parts) == 3 and parts[2] == 'weight':
                        idx = int(parts[1])
                        if idx > classifier_layer_idx:
                            classifier_layer_idx = idx

            in_features = model.classifier[2].in_features

            if classifier_layer_idx == 3:
                model.classifier = nn.Sequential(
                    model.classifier[0],
                    model.classifier[1],
                    nn.Dropout(0.5),
                    nn.Linear(in_features, num_classes)
                )
            else:
                model.classifier[2] = nn.Linear(in_features, num_classes)

            # Load weights
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)

            model.eval()
            model = model.to(self.device)
            self.model = model

            # Find sickle cell class index
            for idx, cls_name in enumerate(self.class_names):
                if 'sickle' in cls_name.lower():
                    self.sickle_cell_class_idx = idx
                    print(f"Found sickle cell class at index {idx}: {cls_name}")
                    break

            # Setup transforms
            self.preprocessor = AdaptiveCellPreprocessing(
                target_size=384,
                normalize_staining=True,
                detect_cell=True,
                fast_mode=False
            )

            self.transform = transforms.Compose([
                transforms.Resize((384, 384)),
                self.preprocessor,
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])

            print("✅ Model loaded successfully!")
            return True

        except Exception as e:
            print(f"❌ Error loading model: {e}")
            import traceback
            traceback.print_exc()
            return False

    def classify_cell(self, image_data, cell_type='WBC'):
        """Classify a single cell image"""
        if self.model is None:
            return {'error': 'Model not loaded'}

        try:
            # Decode base64 image
            if isinstance(image_data, str):
                if ',' in image_data:
                    image_data = image_data.split(',')[1]
                image_bytes = base64.b64decode(image_data)
                image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
            else:
                image = image_data

            # Transform and predict
            input_tensor = self.transform(image).unsqueeze(0).to(self.device)

            with torch.no_grad():
                outputs = self.model(input_tensor)
                probabilities = torch.softmax(outputs, dim=1)
                confidence, predicted_idx = torch.max(probabilities, 1)

            predicted_idx = predicted_idx.item()
            confidence = confidence.item()

            classification = self.class_names[predicted_idx] if self.class_names else f"Class_{predicted_idx}"

            # Check for sickle cell
            is_sickle_cell = False
            sickle_confidence = 0.0
            if self.sickle_cell_class_idx is not None and cell_type == 'RBC':
                sickle_confidence = probabilities[0, self.sickle_cell_class_idx].item()
                is_sickle_cell = sickle_confidence >= self.sickle_cell_confidence_threshold

            return {
                'classification': classification,
                'confidence': confidence,
                'predicted_class_idx': predicted_idx,
                'is_sickle_cell': is_sickle_cell,
                'sickle_confidence': sickle_confidence,
                'all_probabilities': {self.class_names[i]: probabilities[0, i].item()
                                       for i in range(len(self.class_names))} if self.class_names else {}
            }

        except Exception as e:
            return {'error': str(e)}

    def classify_batch(self, images_data, cell_types=None):
        """Classify a batch of cell images"""
        results = []
        for i, img_data in enumerate(images_data):
            cell_type = cell_types[i] if cell_types else 'WBC'
            result = self.classify_cell(img_data, cell_type)
            results.append(result)
        return results


# Initialize global classifier
classifier = ConvNeXtClassifier()
print("✅ Classifier class defined!")

## 5️⃣ Load the Model

In [ ]:
# Load the model
success = classifier.load_model(MODEL_PATH)

if success:
    print("\n" + "="*50)
    print("🎉 Model is ready for classification!")
    print("="*50)
else:
    print("\n❌ Failed to load model. Please check the MODEL_PATH.")

## 6️⃣ Create Flask API Server

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

# Create Flask app
app = Flask(__name__)
CORS(app)

# API Key for basic security (set this in your Render backend)
API_KEY = 'hemalyzer-colab-2024'  # Change this to a secure key!


def check_api_key():
    """Verify API key from request headers"""
    auth_header = request.headers.get('X-API-Key')
    return auth_header == API_KEY


@app.route('/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    return jsonify({
        'status': 'healthy',
        'model_loaded': classifier.model is not None,
        'device': str(classifier.device) if classifier.device else 'not initialized',
        'class_names': classifier.class_names or []
    })


@app.route('/classify', methods=['POST'])
def classify_single():
    """Classify a single cell image"""
    if not check_api_key():
        return jsonify({'error': 'Unauthorized'}), 401

    try:
        data = request.get_json()
        if not data or 'image' not in data:
            return jsonify({'error': 'No image provided'}), 400

        image_data = data['image']
        cell_type = data.get('cell_type', 'WBC')

        result = classifier.classify_cell(image_data, cell_type)
        return jsonify(result)

    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/classify_batch', methods=['POST'])
def classify_batch():
    """Classify a batch of cell images"""
    if not check_api_key():
        return jsonify({'error': 'Unauthorized'}), 401

    try:
        data = request.get_json()
        if not data or 'images' not in data:
            return jsonify({'error': 'No images provided'}), 400

        images = data['images']
        cell_types = data.get('cell_types', ['WBC'] * len(images))

        results = classifier.classify_batch(images, cell_types)
        return jsonify({'results': results})

    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/model_info', methods=['GET'])
def model_info():
    """Get model information"""
    return jsonify({
        'model_loaded': classifier.model is not None,
        'class_names': classifier.class_names or [],
        'num_classes': len(classifier.class_names) if classifier.class_names else 0,
        'sickle_cell_class_idx': classifier.sickle_cell_class_idx,
        'sickle_cell_threshold': classifier.sickle_cell_confidence_threshold,
        'device': str(classifier.device) if classifier.device else 'not initialized'
    })


print("✅ Flask API defined!")

## 7️⃣ Start Server with ngrok Tunnel 🚀

In [ ]:
# Start ngrok tunnel
public_url = ngrok.connect(5000)
print("\n" + "="*60)
print("🌐 NGROK PUBLIC URL (copy this to your Render backend):")
print(f"\n   {public_url}")
print("\n" + "="*60)
print(f"\n📋 Set this in your Render environment variables:")
print(f"   COLAB_MODEL_URL={public_url}")
print(f"   COLAB_API_KEY={API_KEY}")
print("\n" + "="*60)

# Run Flask app in a thread (non-blocking)
def run_flask():
    app.run(port=5000, threaded=True, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("\n✅ Server is running! Keep this notebook open during your demo.")
print("\n🧪 Test the health endpoint:")
print(f"   {public_url}/health")

## 8️⃣ Test the API (Optional)

In [ ]:
import requests

# Test health endpoint
response = requests.get(f"{public_url}/health")
print("Health Check Response:")
print(response.json())

## 9️⃣ Keep Notebook Alive (Run this to prevent timeout)

In [ ]:
import time
from IPython.display import clear_output

# Keep the notebook running
print("🔄 Keeping notebook alive... Press 'Stop' to end.")
print(f"\n🌐 Your API URL: {public_url}")

counter = 0
while True:
    counter += 1
    time.sleep(60)  # Print status every minute
    print(f"⏱️ Server running for {counter} minute(s)... URL: {public_url}")